In [1]:
import pandas as pd
import numpy as np
import pickle
import glob, os
os.chdir("..")

oobasic = pd.read_excel('DATAFILES/oonames.xlsx',sheet_name='eco_oonames')
pickle.dump(oobasic, open('HOME_PICKLE_FILES/oonames.pkl','wb'))
instlist = pd.read_excel('DATAFILES/data_eco_bus.xlsx',sheet_name='listinstitutions')
pickle.dump(instlist, open('HOME_PICKLE_FILES/listinstitutions.pkl','wb'))

In [2]:
def matrixA(datalist, idatalist, journals, institutions):#### to transfer reputation from institutions to journals
    inst = institutions[['institution','acr']].copy()
    for k in range(len(datalist)):
        data = datalist[k]
        datacero = data.drop(['journal'], axis=1)
        datacero['ninst'] = 1/datacero['ninst']
        sj =  datacero.groupby('institution')['ninst'].sum()
        nsj = pd.DataFrame(sj)
        nsj = nsj.reset_index()
        nsj = nsj.rename(columns={'ninst':k})
        nsj = institutions.merge(nsj, on='institution',how='left').fillna(0)
        nsj = nsj[['acr',k]]
        nsj = nsj.sort_values(by=['acr'],ascending=True)
        inst = inst.merge(nsj,on='acr',how='left').fillna(0)
    inst = inst.drop(['institution', 'acr'], axis=1)
    #inst = inst.drop(['institution', 'acr'], axis=1)
    #inst['suma']=np.maximum(inst.sum(axis=1),1)
    #ninst = inst.div(inst.suma,axis=0)#####for use of s_i
    #ninst = ninst.drop(['suma'], axis=1)
    icolsum = np.sum(inst,axis=1)
    #print(inst['pub'])
    icolsum = icolsum.fillna(0)
    icolsum[icolsum < 10] = 500
    return icolsum, inst

In [3]:
def matrixB(datalist, idatalist, journals, institutions):#### to transfer reputation from journals to institutions
    jour= journals[['journal','acr']].copy()
    for k in range(len(idatalist)):
        data = idatalist[k]
        datacero = data[['UT', 'journal']]
        datacero = datacero.drop_duplicates(subset=['UT', 'journal'], keep='first')
        sj =  datacero.groupby('journal')['UT'].count()
        nsj = pd.DataFrame(sj)
        nsj = nsj.reset_index()
        nsj =nsj.rename(columns={'UT':k})
        nsj = journals.merge(nsj, on='journal',how='left').fillna(0)
        nsj = nsj[['acr',k]]
        nsj = nsj.sort_values(by=['acr'],ascending=True)
        jour = jour.merge(nsj,on='acr',how='left').fillna(0)
    jour = jour.drop(['journal', 'acr'], axis=1)
    #jour['suma']=np.maximum(jour.sum(axis=1),1)
    #njour = jour.div(jour.suma,axis=0)##### for use of s_i
    #njour = njour.drop(['suma'], axis=1)
    jcolsum = np.sum(jour,axis=1)
    jcolsum = jcolsum.fillna(0)
    jcolsum[jcolsum < 25] = 25
    #colsum = np.maximum(jcolsum,25)
    return jcolsum, jour

In [4]:
def updatescores(journals, institutions,  A, B):### computing the first eigenvector of BA
    v = institutions['iscore'].to_numpy()
    oldnorma = 10
    norma = 1
    while np.abs(oldnorma-norma) > 0.000000001:
        oldnorma = norma
        v = np.matmul(np.matmul(B,A),v)
        norma = np.linalg.norm(v)
        v = v/norma
        print(np.round(norma,10)) ### To witness how the convergence to the first eigenvalue takes place, we follow the value of the norm of the vector.
    w = np.matmul(A,v)### the first eigenvector of AB
    return v, w

In [5]:
jourdatalist= []
instdatalist = []
fields = {'eco_bus'}
#fields = {'man'}
for field in fields:
    print(field)
    #get the files needed for the algorithm from two folders: DATAFILES AND HOME_PICKLE_FILES
    #HOME_PICKLE_FILES STORES THE INFORMATION GATHERED FROM THE CITATION REPORTS OF JOURNALS AND INSTITUTIONS
    #OTHER JUNYPER NOTEBOOKS PREPARE THE INFORMATION AND STORE IT AS PICKLE FILES (EASY TO MANAGE WITHIN PYTHON)
    file = 'DATAFILES/data_'+ field+ '.xlsx'
    xl = pd.ExcelFile(file)
    d = {} # your dict.
    for sheet in xl.sheet_names:
        d[f'{sheet}']= pd.read_excel(xl,sheet_name=sheet)
        pickle.dump(d[f'{sheet}'], open('HOME_PICKLE_FILES/' + sheet + '.pkl','wb'))
    idatalist = pd.read_pickle('HOME_PICKLE_FILES/idatalist.pkl')
    datalist = pd.read_pickle('HOME_PICKLE_FILES/jdatalist.pkl')
    institutions = pd.read_pickle("HOME_PICKLE_FILES/institutions.pkl")
    journals = pd.read_pickle("HOME_PICKLE_FILES/journals.pkl")
    
    icolsum, inst = matrixA(datalist, idatalist, journals, institutions)### CITATION MATRIX FROM INSTITUTIONS TO JOURNALS
    jcolsum, jour = matrixB(datalist, idatalist, journals, institutions)### CITATION MATRIX FROM JOURNALS TO INSTITUTIONS
    ninst = inst / jcolsum
    ninst = ninst.T
    A = ninst.to_numpy()
    njour = jour / icolsum
    njour = njour.T
    B = njour.to_numpy()
    v, w = updatescores(journals, institutions, A, B)### SIZE-DEPENDENT INFLUENCE EIGENVECTORS FOR INSTITUTIONS AND JOURNALS
    
    df = pd.DataFrame(v, columns=['iscore'])
    inst = institutions[['institution','pub','acr']].copy()
    inst = inst.join(df)
    #inst['iscore'] = inst['iscore']/inst['pub']#### SIZE-INDEPENDENT INFLUENCE PER PAPER
    a = 7/( inst['iscore'].max()-inst['iscore'].min())
    inst['iscore'] =  1+a*(inst['iscore']-inst['iscore'].min())
 
    df = pd.DataFrame(w, columns=['journal_score'])
    jour = journals[['journal','acr']].copy()
    jour = journals[['journal','acr','pub']].copy()
    jour = jour.join(df)
    #jour['journal_score'] = jour['journal_score']/jour['pub']#### SIZE-INDEPENDENT INFLUENCE PER PAPER
    a = 9.99/( jour['journal_score'].max()-jour['journal_score'].min())
    jour['journal_score'] =  0.01+a*(jour['journal_score']-jour['journal_score'].min())

    inst = inst.sort_values(by=['iscore'],ascending=False)
    jour = jour.sort_values(by=['journal_score'],ascending=False)
    jourdatalist.append(jour)
    instdatalist.append(inst)   

eco_bus
202.2168045958
0.9374951932
0.9149687288
0.9161397891
0.9172204732
0.9176102573
0.9177316682
0.9177679818
0.9177787079
0.9177818635
0.9177827906
0.9177830629
0.9177831429
0.9177831664
0.9177831733
0.9177831753
0.9177831759


In [6]:
filename = 'old_python_files_for_jirc/jirc_journal_institution_results.xlsx'
with pd.ExcelWriter(filename) as writer:
    for k, field in zip(range(len(jourdatalist)),fields):
        jourdatalist[k].to_excel(writer,sheet_name= 'journals')
        instdatalist[k].to_excel(writer,sheet_name= 'institutions')

In [7]:
pwd

'C:\\Users\\usuario\\Dropbox\\ECONOMICS_BUSINESS'

os.chdir("old_python_files_for_jirc")

pd.DataFrame(A).to_excel('matriza.xlsx')

In [15]:
pd.DataFrame(icolsum).to_excel('old_python_files_for_jirc/matriza.xlsx')

In [9]:
inst

,institution,pub,acr,iscore
40,University of Maryland College Park,128,ABO,8.000000
309,University of Arkansas Fayetteville,99,BEE,4.009312
240,"City St Georges, University of London",125,BBL,3.671177
235,heSam Universite,98,BBG,3.167551
340,Baruch College (CUNY),57,BFK,2.031654
...,...,...,...,...
363,Zhejiang Gongshang University,57,BGH,1.001348
378,Vilnius Gediminas Technical University,48,BGX,1.001265
467,Shanghai Lixin University of Accounting & Finance,45,CGL,1.000942
576,University of Novi Sad,42,DEB,1.000476


In [10]:
icolsum

0      1369.697425
1      1236.911985
2      1797.657584
3       302.933775
4      1853.934363
          ...     
595    2567.529942
596     267.467002
597     221.294048
598     500.000000
599     635.342297
Length: 600, dtype: float64

In [11]:
icolsum = np.where(icolsum == 100, inst['pub']*5,icolsum)

In [12]:
inst['pub']

40     128
309     99
240    125
235     98
340     57
      ... 
363     57
378     48
467     45
576     42
595     58
Name: pub, Length: 600, dtype: int64

In [13]:
icolsum

array([1369.6974248 , 1236.91198524, 1797.65758408,  302.93377456,
       1853.93436286, 1875.68612776, 1181.52071817, 1331.9020202 ,
       1033.03788989, 1610.09070374, 1212.83054723,  881.04837385,
       1103.94633977,  624.15896603, 1340.24065379, 1384.39133922,
       1121.31653069,  752.21558997,  487.12220557, 1512.88238151,
       1053.36948884, 1247.27712288, 1289.54642857,  479.91909479,
        716.88913864,  974.51397214,  741.46421356, 1245.68438506,
        769.5713703 ,  684.98668554, 1279.27705628,  273.34517705,
       1053.39343157, 1072.71858974, 2117.52536353,  500.        ,
        635.16896159,  556.95599678, 1309.02762238, 1148.97644578,
         11.95238095,  953.13677711, 1939.51669442,  761.28855034,
        943.22753635, 1233.32997558,  369.0497114 ,  899.18638584,
        790.79375624,  854.14374514, 1365.56771284,  676.66789322,
        476.01793484,  415.47657343,  500.        , 1170.54163059,
       1223.34564047,  562.60443723,  498.61156899,  717.57874